In [3]:
from langchain.messages import  SystemMessage,HumanMessage,AIMessage
from langgraph.graph.message import add_messages

left = [
    SystemMessage(content="你是一个专业的翻译",id = '1'),
    HumanMessage(content="你好",id = '2'),
    AIMessage(content="你好，我是专业的翻译",id = '3'),
]

right = [
    HumanMessage(content="我是老王,你是小王",id = '2'),
    AIMessage(content="好的我记住了",id = '3'),
    HumanMessage(content="你是谁?",id = '4'),
    AIMessage(content="我是小王",id = '5'),
    HumanMessage(content="你是小王吗?",id = '1'),
]
merged = add_messages(left,right)     # 合并消息`  旧信息被覆盖
print(merged)


[HumanMessage(content='你是小王吗?', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='我是老王,你是小王', additional_kwargs={}, response_metadata={}, id='2'), AIMessage(content='好的我记住了', additional_kwargs={}, response_metadata={}, id='3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你是谁?', additional_kwargs={}, response_metadata={}, id='4'), AIMessage(content='我是小王', additional_kwargs={}, response_metadata={}, id='5', tool_calls=[], invalid_tool_calls=[])]


In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class OverAllState(TypedDict):
    logs: list[str]      #没有使用annotated 后面就会直接覆盖
    id: str

def node_a(state: OverAllState):
    return {
        "logs": ["node_a"],
        "id": "node_a"
    }

def node_b(state: OverAllState):
    return {
        "logs": ["node_b"],
        "id": "node_b"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
print(result)

============================== -> result <- ==============================
{'logs': ['node_b'], 'id': 'node_b'}


In [1]:
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END


# 1. 定义状态
from langchain_core.messages import AnyMessage

class OverAllState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# 或者直接：
# MessagesState 已经内置 add_messages Reducer，是 LangGraph 管理对话消息的标准方式。
# from langgraph.graph import MessagesState
#
# class OverAllState(MessagesState):
#     pass


# 2. 定义节点
def node_a(state: OverAllState):
    return {
        "messages": [
            HumanMessage(content="你好，我是老王", id="1"),
            AIMessage(content="你好，老王", id="2")
        ]
    }


def node_b(state: OverAllState):
    return {
        "messages": [
            AIMessage(content="你好，我是小王", id="2"),  # 覆盖 ID=2
            HumanMessage(content="你是谁？", id="3")     # 新增 ID=3
        ]
    }


# 3. 构建 Graph
builder = StateGraph(OverAllState)

builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()


# 4. 执行
result = graph.invoke({
    "messages": []
})

for msg in result["messages"]:
    print(f"ID={msg.id}, content={msg.content}")

ID=1, content=你好，我是老王
ID=2, content=你好，我是小王
ID=3, content=你是谁？
